# Notebook 01: Data Profiling & Exploration (Bronze Layer)
This notebook profiles the raw UK Housing dataset loaded into the Bronze Parquet layer, inspecting schema, null distributions, summary statistics, and price distribution without invoking driver-bound `.collect()` calls.

In [ ]:
import sys
import json
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# Add src to python path for utils
sys.path.append('../src')
from utils import create_spark_session, get_project_root, save_json

# Cell 1: Initialize SparkSession & Load Bronze Parquet
spark = create_spark_session(app_name="01-profiling")
root_dir = get_project_root()
bronze_path = root_dir / "data" / "bronze"
df_bronze = spark.read.parquet(str(bronze_path))
print(f"Loaded Bronze data from: {bronze_path}")

In [ ]:
# Cell 2: Display Schema
df_bronze.printSchema()

In [ ]:
# Cell 3: Count Rows & Display Sample Rows
total_rows = df_bronze.count()
print(f"Total Bronze Rows: {total_rows:,}")
df_bronze.show(10, truncate=False)

In [ ]:
# Cell 4: Missing Value Counts Per Column (Spark SQL Aggregations)
null_exprs = [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_bronze.columns]
null_df = df_bronze.select(null_exprs)
null_df.show(truncate=False)

In [ ]:
# Cell 5: Duplicate Row Count
distinct_rows = df_bronze.dropDuplicates().count()
duplicate_count = total_rows - distinct_rows
print(f"Exact Duplicate Rows: {duplicate_count:,} ({duplicate_count / total_rows * 100:.2f}%)")

In [ ]:
# Cell 6: Data Types Summary
dtypes_summary = [{"column": col_name, "dataType": dtype} for col_name, dtype in df_bronze.dtypes]
spark.createDataFrame(dtypes_summary).show(truncate=False)

In [ ]:
# Cell 7: Price Column Statistics (mean, median, std, min, max, quartiles)
# Find price column name (case-insensitive)
price_col = [c for c in df_bronze.columns if c.lower() == 'price'][0]
df_numeric_price = df_bronze.withColumn("price_num", F.col(price_col).cast(DoubleType()))

price_stats = df_numeric_price.select(
    F.count("price_num").alias("count"),
    F.mean("price_num").alias("mean"),
    F.stddev("price_num").alias("stddev"),
    F.min("price_num").alias("min"),
    F.max("price_num").alias("max"),
    F.percentile_approx("price_num", 0.25).alias("q1"),
    F.percentile_approx("price_num", 0.50).alias("median"),
    F.percentile_approx("price_num", 0.75).alias("q3")
)
price_stats.show(truncate=False)

In [ ]:
# Cell 8: Price Distribution (Histogram Bins using Spark Aggegration)
price_bins = df_numeric_price.select(
    F.when(F.col("price_num") < 100000, "< 100k")
    .when((F.col("price_num") >= 100000) & (F.col("price_num") < 250000), "100k - 250k")
    .when((F.col("price_num") >= 250000) & (F.col("price_num") < 500000), "250k - 500k")
    .when((F.col("price_num") >= 500000) & (F.col("price_num") < 1000000), "500k - 1M")
    .otherwise("> 1M").alias("price_range")
).groupBy("price_range").count().orderBy("count", ascending=False)

price_bins.show(truncate=False)

In [ ]:
# Cell 9: Outlier Detection (IQR method)
q_row = df_numeric_price.select(
    F.percentile_approx("price_num", 0.25).alias("q1"),
    F.percentile_approx("price_num", 0.75).alias("q3")
).first()

q1 = q_row["q1"]
q3 = q_row["q3"]
iqr = q3 - q1
lower_bound = max(0, q1 - 1.5 * iqr)
upper_bound = q3 + 1.5 * iqr

outliers_count = df_numeric_price.filter((F.col("price_num") < lower_bound) | (F.col("price_num") > upper_bound)).count()
print(f"IQR: {iqr:,.2f} | Lower Bound: {lower_bound:,.2f} | Upper Bound: {upper_bound:,.2f}")
print(f"Outlier Count (IQR): {outliers_count:,} ({outliers_count / total_rows * 100:.2f}%)")

In [ ]:
# Cell 10: Save Profiling Results to results/profiling_results.json
stats_dict = price_stats.first().asDict()
profiling_results = {
    "total_rows": total_rows,
    "columns": df_bronze.columns,
    "duplicate_count": duplicate_count,
    "price_statistics": {
        "count": stats_dict["count"],
        "mean": float(stats_dict["mean"]),
        "stddev": float(stats_dict["stddev"]),
        "min": float(stats_dict["min"]),
        "max": float(stats_dict["max"]),
        "q1": float(stats_dict["q1"]),
        "median": float(stats_dict["median"]),
        "q3": float(stats_dict["q3"])
    },
    "iqr_outliers": {
        "iqr": float(iqr),
        "lower_bound": float(lower_bound),
        "upper_bound": float(upper_bound),
        "outlier_count": outliers_count
    }
}

save_json(profiling_results, root_dir / "results" / "profiling_results.json")
print("Profiling completed and saved successfully!")